# Módulo 3 — Benchmark de familias de detección de anomalías

El **Módulo 2** cubre tres enfoques no supervisados (aislamiento, densidad local, z-score robusto por columna) más un autoencoder. Este módulo cierra el mapa: agrega las familias que faltaban y las compara a todas en el mismo split, con el mismo escalado y las mismas métricas.

La pregunta no es solo *"cuál gana"*. Sin etiquetas no se puede elegir el mejor detector antes de desplegarlo, así que las dos preguntas realmente accionables son:

1. **¿Qué familias son complementarias?** Si dos detectores ordenan las transacciones casi igual, tener los dos no aporta nada. Se mide con la correlación de Spearman entre sus *anomaly scores*.
2. **¿Cuánto cuesta cada punto de PR-AUC?** Se cronometran ajuste y scoring por separado: en producción el scoring corre por transacción y el ajuste una vez al día.

Y una tercera, que es la respuesta natural a la primera: **si ningún detector domina, ¿conviene combinarlos?** (`src/unsupervised/ensemble.py`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.unsupervised.benchmark import (
    evaluate_all,
    plot_correlation,
    plot_pr_curves,
    plot_ranking,
    run_detectors,
    score_correlation,
    summary_table,
)
from src.unsupervised.ensemble import build_ensembles
from src.unsupervised.loader import get_unsupervised_data

## 1. Datos y escalado

Exactamente el mismo split que el Módulo 2 (`get_unsupervised_data`, semilla fija): entrenamiento con transacciones **100% normales**, prueba mixta. Comparar detectores tiene sentido solo si todos ven los mismos datos.

In [ ]:
X_train, X_test, y_test = get_unsupervised_data()

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train (solo normales): {X_train.shape}")
print(f"Test (mixto): {X_test.shape} — fraude: {int(y_test.sum())} ({y_test.mean():.2%})")

## 2. Las once familias

| Detector | Familia | Qué anomalía detecta bien |
|---|---|---|
| Isolation Forest | Aislamiento | Puntos que se separan con pocas particiones aleatorias |
| Local Outlier Factor | Densidad local | Puntos en zonas menos densas que su vecindario |
| kNN (k-ésima distancia) | Distancia global | Puntos lejos de cualquier vecindario denso |
| MAD-z | Estadístico por feature | Valores extremos en una columna |
| HBOS | Estadístico por feature | Ídem, con histogramas; descompone el score por columna |
| ECOD | Colas de la CDF empírica | Colas extremas, sin un solo hiperparámetro |
| Mahalanobis robusto (MCD) | Covarianza robusta | Correlaciones rotas entre features |
| Gaussian Mixture | Densidad paramétrica | Huecos de baja probabilidad entre modos |
| One-Class SVM (Nyström) | Frontera con kernel | Filas fuera de la envolvente de lo normal |
| PCA (reconstrucción) | Reconstrucción lineal | Filas fuera del subespacio principal |
| Autoencoder (ReLU) | Reconstrucción no lineal | Ídem, capturando interacciones no lineales |

PCA y autoencoder están los dos a propósito: PCA es la **ablación** del autoencoder. Si la red no supera al PCA, su no-linealidad no está aportando nada en este dataset y no vale su costo.

In [ ]:
outputs = run_detectors(X_train_scaled, X_test_scaled)

## 3. Ensembles

Tres formas de combinar scores que viven en escalas incompatibles (una distancia de Mahalanobis no es comparable con una log-verosimilitud):

- **promedio de rangos** — descarta la magnitud, conserva solo el orden;
- **promedio z** — estandariza cada detector y promedia, conservando *cuánto* de anómalo;
- **máximo z** — basta con que un detector grite fuerte, útil si cada tipo de fraude lo ve una sola familia.

Su costo de ajuste es cero: operan sobre los scores ya calculados.

In [ ]:
base_scores = {name: output["scores"] for name, output in outputs.items()}

for name, scores in build_ensembles(base_scores).items():
    outputs[name] = {"scores": scores, "fit_seconds": 0.0, "score_seconds": 0.0}

results = evaluate_all(outputs, y_test)

## 4. Tabla comparativa

PR-AUC como métrica principal (con un desbalance de ~0.1% el ROC-AUC se ve optimista casi siempre), Precision@100 como la lectura operativa: *de las 100 transacciones más anómalas que revisaría un analista, cuántas son fraude real*.

In [ ]:
summary_table(results)

## 5. Ranking por familia

In [ ]:
fig = plot_ranking(results, output_path=None)
plt.show()

## 6. ¿Qué detectores son redundantes?

Correlación de Spearman entre los *rankings* de anomalía. Un par con correlación cercana a 1 ordena las transacciones casi igual: mantener los dos duplica el costo sin agregar cobertura. Los pares con correlación baja son los que hacen que valga la pena un ensemble.

In [ ]:
corr = score_correlation(results)

fig = plot_correlation(corr, output_path=None)
plt.show()

In [ ]:
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()

print("Pares más redundantes (Spearman > 0.9):")
print(upper[upper > 0.9].sort_values(ascending=False).to_string() or "  ninguno")

print("\nPares más complementarios (menor correlación):")
print(upper.sort_values().head(5).to_string())

## 7. Curvas Precision-Recall de los mejores

In [ ]:
fig = plot_pr_curves(results, y_test, output_path=None)
plt.show()

## 8. Conclusiones

- **Ninguna familia domina en todo el rango de recall.** Las curvas PR se cruzan: el detector que da mejor precisión en las primeras 50 alertas no siempre es el que más fraude recupera en total. La elección depende de la capacidad de revisión del equipo, no solo del PR-AUC.
- **La matriz de correlación justifica —o descarta— el ensemble.** Detectores muy correlacionados son intercambiables; el valor de combinar aparece solo cuando hay familias que ordenan distinto.
- **El costo importa tanto como la métrica.** Los detectores de coste lineal (HBOS, ECOD, MAD-z, PCA) puntúan en milisegundos y no requieren un índice de vecinos en memoria; los basados en distancias (LOF, kNN) pagan cada predicción contra todo el set de entrenamiento, lo que condiciona si son viables en un flujo transaccional en línea.
- **PCA vs. autoencoder** es la comparación honesta que suele faltar: la ablación lineal dice si la red neuronal está ganando algo real o solo agregando complejidad.

Las métricas de cada corrida quedan persistidas en `data/processed/metrics.duckdb` (tabla `benchmark_metrics`) para comparar entre ejecuciones.